In [0]:
from pyspark.sql.functions import (
    current_timestamp, lit, col, to_date,
    sum as spark_sum, current_date, to_timestamp
)
from pyspark.sql.window import Window

VOLUME_PATH = "/Volumes/de_workspace26/ecommerce_pawan/raw_files"
CATALOG     = "de_workspace26"
SCHEMA_B    = f"{CATALOG}.bronze_pawan"
SCHEMA_S    = f"{CATALOG}.silver_pawan"
SCHEMA_G    = f"{CATALOG}.gold_pawan"

print("Constants set.")
print("Volume path :", VOLUME_PATH)

In [0]:
print("=" * 60)
print("OUTPUT 1 — BRONZE DELTA TABLES")
print("=" * 60)

# Row counts
bronze_tables = ["orders", "customers", "products", "orders_stream"]
for table in bronze_tables:
    count = spark.read.table(f"{SCHEMA_B}.{table}").count()
    print(f"\n  {SCHEMA_B}.{table} — {count} rows")

# Metadata columns check
print("\n=== Metadata columns in bronze.orders ===")
spark.sql(f"""
    SELECT
        order_id,
        _ingested_at,
        _source_file
    FROM {SCHEMA_B}.orders
    LIMIT 5
""").show(truncate=False)

# NOT NULL constraint check
print("=== Constraints on bronze.orders ===")
spark.sql(f"""
    SHOW TBLPROPERTIES {SCHEMA_B}.orders
""").show(truncate=False)

In [0]:
print("=== Transaction History — bronze.orders ===")
spark.sql(f"""
    DESCRIBE HISTORY {SCHEMA_B}.orders
""").select(
    "version",
    "timestamp",
    "operation",
    "operationParameters"
).show(10, truncate=False)

print("=== Transaction History — bronze.customers ===")
spark.sql(f"""
    DESCRIBE HISTORY {SCHEMA_B}.customers
""").select(
    "version",
    "timestamp",
    "operation"
).show(5, truncate=False)

print("=== Transaction History — bronze.products ===")
spark.sql(f"""
    DESCRIBE HISTORY {SCHEMA_B}.products
""").select(
    "version",
    "timestamp",
    "operation"
).show(5, truncate=False)

print("✅ Output 1 verified — Bronze tables with metadata and history.")

In [0]:
print("=" * 60)
print("OUTPUT 2 — SILVER LAYER")
print("=" * 60)

# silver.orders row count
silver_orders_count = spark.read.table(f"{SCHEMA_S}.orders").count()
print(f"\n  {SCHEMA_S}.orders — {silver_orders_count} rows")  # 200

# Deduplication check
bronze_count = spark.read.table(f"{SCHEMA_B}.orders").count()
print(f"\n  Deduplication:")
print(f"     Bronze rows : {bronze_count}")          # 205
print(f"     Silver rows : {silver_orders_count}")   # 200
print(f"     Duplicates removed : {bronze_count - silver_orders_count}")  # 5

# Enriched columns check
print("\n=== Enriched columns in silver.orders ===")
spark.sql(f"""
    SELECT
        order_id, customer_id, city, loyalty_tier,
        product_name, category,
        revenue, cumulative_revenue,
        region, order_date
    FROM {SCHEMA_S}.orders
    LIMIT 5
""").show(truncate=False)

# Partition check
print("=== Partition counts by region ===")
spark.sql(f"""
    SELECT region, COUNT(*) AS row_count
    FROM   {SCHEMA_S}.orders
    GROUP BY region
    ORDER BY region
""").show()

In [0]:
# SCD Type 2 check
print("=== SCD Type 2 — silver.customers ===")
silver_cust_count = spark.read.table(f"{SCHEMA_S}.customers").count()
print(f"  Total rows : {silver_cust_count}")   # 30

spark.sql(f"""
    SELECT is_current, COUNT(*) AS count
    FROM   {SCHEMA_S}.customers
    GROUP BY is_current
""").show()
# is_current=true  → 20
# is_current=false → 10

print("=== SCD Type 2 sample — CUST01 to CUST03 ===")
spark.sql(f"""
    SELECT
        customer_id, loyalty_tier, is_current,
        effective_start_date, effective_end_date
    FROM   {SCHEMA_S}.customers
    WHERE  customer_id IN ('CUST01','CUST02','CUST03')
    ORDER BY customer_id, effective_start_date
""").show(truncate=False)

# CDF check
print("=== CDF enabled on silver.orders ===")
spark.sql(f"""
    SHOW TBLPROPERTIES {SCHEMA_S}.orders
""").filter("key = 'delta.enableChangeDataFeed'").show(truncate=False)

print("=== CDF change log ===")
spark.sql(f"""
    SELECT _change_type, order_id, status, region
    FROM   table_changes('{SCHEMA_S}.orders', 1)
    LIMIT  10
""").show(truncate=False)

print("✅ Output 2 verified — Silver layer complete.")

In [0]:
print("=" * 60)
print("OUTPUT 3 — GOLD VIEWS")
print("=" * 60)

# List all gold views
print("=== Views in gold_pawan schema ===")
spark.sql(f"SHOW VIEWS IN {SCHEMA_G}").show(truncate=False)

# monthly_revenue_by_region
print("\n=== gold.monthly_revenue_by_region ===")
spark.sql(f"""
    SELECT *
    FROM   {SCHEMA_G}.monthly_revenue_by_region
    ORDER BY year, month, region
""").show(20, truncate=False)

# top_products
print("\n=== gold.top_products — Top 10 ===")
spark.sql(f"""
    SELECT
        product_name, category,
        total_revenue, total_quantity,
        rank_in_category
    FROM   {SCHEMA_G}.top_products
    ORDER BY total_revenue DESC
    LIMIT  10
""").show(truncate=False)

# Revenue summary
print("\n=== Total revenue by region ===")
spark.sql(f"""
    SELECT
        region,
        SUM(total_revenue) AS total_revenue,
        SUM(total_orders)  AS total_orders
    FROM   {SCHEMA_G}.monthly_revenue_by_region
    GROUP BY region
    ORDER BY total_revenue DESC
""").show()

print("✅ Output 3 verified — Gold views accessible.")

In [0]:
print("=" * 60)
print("OUTPUT 4 — STREAMING PIPELINE")
print("=" * 60)

# live_orders count
live_count = spark.read.table(f"{SCHEMA_G}.live_orders").count()
print(f"\n  {SCHEMA_G}.live_orders — {live_count} rows")   # 235

# Checkpoint verification
print("\n=== Checkpoint files ===")
try:
    ck_files = dbutils.fs.ls(CHECKPOINT_LIVE)
    for f in ck_files:
        print(f"  - {f.name}")
    print(f"✅ Checkpoint exists at : {CHECKPOINT_LIVE}")
except Exception as e:
    print(f"❌ Checkpoint not found : {e}")

# Idempotency verification
print("\n=== Idempotency check ===")
print(f"  Row count before restart : {count_before}")
print(f"  Row count after  restart : {count_after}")

if count_before == count_after:
    print("  ✅ PASSED — No duplicate rows written.")
else:
    print(f"  ❌ FAILED — {count_after - count_before} extra rows.")

# Preview
print("\n=== live_orders sample ===")
spark.read.table(f"{SCHEMA_G}.live_orders") \
     .orderBy("order_id") \
     .show(10, truncate=False)

print("✅ Output 4 verified — Streaming pipeline complete.")

In [0]:
print("=" * 60)
print("OUTPUT 5 — WORKFLOW DAG")
print("=" * 60)

print("""
  Job name   : ecommerce_pipeline_job
  Schedule   : 0 6 * * * (Daily at 06:00 AM UTC)
  Notification: Email on Failure

  DAG Structure (sequential):
  ┌──────────────────────────────────────┐
  │  Task 1: ingest_bronze               │
  │  → Retries : 2 (5 min gap)           │
  │  → Cluster : ecommerce-job-cluster   │
  └──────────────────┬───────────────────┘
                     ↓
  ┌──────────────────────────────────────┐
  │  Task 2: transform_silver            │
  │  → Depends on : ingest_bronze        │
  │  → Cluster    : ecommerce-job-cluster│
  └──────────────────┬───────────────────┘
                     ↓
  ┌──────────────────────────────────────┐
  │  Task 3: build_gold                  │
  │  → Depends on : transform_silver     │
  │  → Cluster    : ecommerce-job-cluster│
  └──────────────────┬───────────────────┘
                     ↓
  ┌──────────────────────────────────────┐
  │  Task 4: run_streaming               │
  │  → Depends on : build_gold           │
  │  → Cluster    : ecommerce-job-cluster│
  └──────────────────────────────────────┘

  Cluster config:
  → Runtime          : Latest LTS
  → Auto-termination : 30 minutes
""")
print("✅ Output 5 verified — Workflow DAG configured.")

In [0]:
print("=" * 60)
print("OUTPUT 6 — UNITY CATALOG GOVERNANCE")
print("=" * 60)

# Show all tables
print("=== Tables in bronze_pawan ===")
spark.sql(f"SHOW TABLES IN {SCHEMA_B}").show(truncate=False)

print("=== Tables in silver_pawan ===")
spark.sql(f"SHOW TABLES IN {SCHEMA_S}").show(truncate=False)

print("=== Views in gold_pawan ===")
spark.sql(f"SHOW VIEWS IN {SCHEMA_G}").show(truncate=False)

# Column mask verification
print("=== Column mask on customers.email ===")
spark.sql(f"""
    SELECT
        customer_id,
        name,
        email
    FROM {SCHEMA_S}.customers
    WHERE is_current = true
    LIMIT 5
""").show(truncate=False)
# Without pii_access: al*******************

# Row filter verification
print("=== Row filter on silver.orders ===")
spark.sql(f"""
    SHOW TBLPROPERTIES {SCHEMA_S}.orders
""").filter("key LIKE '%rowFilter%'").show(truncate=False)

print("✅ Output 6 verified — Unity Catalog governance applied.")

In [0]:
%sql
-- bonus

In [0]:
print("=" * 60)
print("BONUS 1 — TIME TRAVEL")
print("=" * 60)

# Show all versions
print("=== Full transaction history ===")
spark.sql(f"""
    DESCRIBE HISTORY {SCHEMA_B}.orders
""").select(
    "version", "timestamp", "operation"
).show(truncate=False)

# Query version 0 (original ingestion)
print("=== Version 0 — Original ingestion ===")
v0_count = spark.sql(f"""
    SELECT COUNT(*) AS row_count
    FROM   {SCHEMA_B}.orders VERSION AS OF 0
""").collect()[0]["row_count"]
print(f"  Rows at version 0 : {v0_count}")   # 205

# Query current version
current_count = spark.read.table(f"{SCHEMA_B}.orders").count()
print(f"  Rows at current   : {current_count}")

# Show data at version 0
print("\n=== Sample rows at version 0 ===")
spark.sql(f"""
    SELECT order_id, status, _ingested_at
    FROM   {SCHEMA_B}.orders VERSION AS OF 0
    LIMIT  5
""").show(truncate=False)

# Query by timestamp
print("=== Latest version details ===")
latest = spark.sql(f"""
    DESCRIBE HISTORY {SCHEMA_B}.orders LIMIT 1
""").collect()[0]
print(f"  Latest version   : {latest['version']}")
print(f"  Latest timestamp : {latest['timestamp']}")

print("✅ Bonus 1 complete — Time Travel verified.")

In [0]:
print("=" * 60)
print("BONUS 2 — VACUUM")
print("=" * 60)

# Show history before vacuum
print("=== History before VACUUM ===")
spark.sql(f"""
    DESCRIBE HISTORY {SCHEMA_S}.orders
""").select(
    "version", "timestamp", "operation"
).show(truncate=False)

# Dry run first — see what would be deleted
print("\n=== DRY RUN — Files that would be deleted ===")
spark.sql(f"""
    VACUUM {SCHEMA_S}.orders DRY RUN
""").show(truncate=False)

# Disable retention check for demo (default is 7 days)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# Run VACUUM — retain 0 hours to see full effect in demo
print("\n=== Running VACUUM RETAIN 0 HOURS (demo only) ===")
spark.sql(f"""
    VACUUM {SCHEMA_S}.orders RETAIN 0 HOURS
""").show(truncate=False)

# Re-enable retention check
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

# Verify table still works after vacuum
post_vacuum_count = spark.read.table(f"{SCHEMA_S}.orders").count()
print(f"\n  Rows after VACUUM : {post_vacuum_count}")   # 200 — unchanged

# Show history after vacuum
print("\n=== History after VACUUM ===")
spark.sql(f"""
    DESCRIBE HISTORY {SCHEMA_S}.orders
""").select(
    "version", "timestamp", "operation"
).show(truncate=False)

print("✅ Bonus 2 complete — VACUUM run on silver.orders.")

In [0]:
import time

print("=" * 60)
print("BONUS 3 — PHOTON ENGINE BENCHMARK")
print("=" * 60)

print("""
  To enable Photon:
  1. Go to Compute → Edit your cluster
  2. Check 'Enable Photon Acceleration'
  3. Restart the cluster
  4. Re-run this cell to compare
""")

# Benchmark query 1 — monthly revenue aggregation
print("=== Query 1: Monthly Revenue Aggregation ===")
start = time.time()
result1 = spark.sql(f"""
    SELECT *
    FROM   {SCHEMA_G}.monthly_revenue_by_region
""").count()
elapsed1 = time.time() - start
print(f"  Rows returned : {result1}")
print(f"  Elapsed time  : {elapsed1:.3f} seconds")

# Benchmark query 2 — top products ranking
print("\n=== Query 2: Top Products with RANK() ===")
start = time.time()
result2 = spark.sql(f"""
    SELECT *
    FROM   {SCHEMA_G}.top_products
""").count()
elapsed2 = time.time() - start
print(f"  Rows returned : {result2}")
print(f"  Elapsed time  : {elapsed2:.3f} seconds")

# Benchmark query 3 — silver orders full scan
print("\n=== Query 3: Silver Orders Full Scan ===")
start = time.time()
result3 = spark.sql(f"""
    SELECT
        region,
        loyalty_tier,
        category,
        SUM(revenue)            AS total_revenue,
        AVG(cumulative_revenue) AS avg_cumulative,
        COUNT(order_id)         AS total_orders
    FROM {SCHEMA_S}.orders
    GROUP BY region, loyalty_tier, category
    ORDER BY total_revenue DESC
""").count()
elapsed3 = time.time() - start
print(f"  Rows returned : {result3}")
print(f"  Elapsed time  : {elapsed3:.3f} seconds")

print(f"""
  ┌────────────────────────────────────────────┐
  │           PHOTON BENCHMARK RESULTS         │
  ├──────────────────────────┬─────────────────┤
  │ Query                    │ Time (seconds)  │
  ├──────────────────────────┼─────────────────┤
  │ Monthly Revenue          │ {elapsed1:.3f}           │
  │ Top Products RANK()      │ {elapsed2:.3f}           │
  │ Silver Orders Full Scan  │ {elapsed3:.3f}           │
  └──────────────────────────┴─────────────────┘

  Run again with Photon enabled to compare times.
  Photon typically shows 2-5x improvement on
  aggregation and scan heavy queries.
""")
print("✅ Bonus 3 complete — Photon benchmark recorded.")